# Variational classifier

Train a tiny parity classifier with parameter-shift gradients and compare predictions and loss.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [2]:
features = np.asarray([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
labels = np.asarray([1.0, -1.0, -1.0, 1.0])

def make_model(device):
    @qml.qnode(device, diff_method="parameter-shift")
    def circuit(x, weights):
        qml.RX(np.pi * x[0], wires=0)
        qml.RX(np.pi * x[1], wires=1)
        qml.RY(weights[0], wires=0)
        qml.RY(weights[1], wires=1)
        qml.CNOT(wires=[0, 1])
        qml.RY(weights[2], wires=1)
        return qml.expval(qml.Z(1))
    return circuit

def train(model):
    weights = pnp.array([0.2, -0.1, 0.3], requires_grad=True)
    def loss(current):
        predictions = pnp.stack([model(row, current) for row in features])
        return pnp.mean((predictions - labels) ** 2)
    trace = []
    for _ in range(6):
        trace.append(float(loss(weights)))
        weights = weights - 0.18 * qml.grad(loss)(weights)
    predictions = np.asarray([model(row, weights) for row in features], dtype=float)
    return np.asarray(trace), predictions

reference_model = make_model(qml.device("default.qubit", wires=2))
reference, reference_ms, _ = benchmark(lambda: train(reference_model), repeats=2)
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_model = make_model(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: train(mettleq_model), repeats=2)
trace_error = max_abs_error(reference[0], candidate[0])
prediction_error = max_abs_error(reference[1], candidate[1])
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/08_variational_classifier.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="training trace and predictions atol=8e-5",
    passed=max(trace_error, prediction_error) <= 8e-5,
    exact_match=bool(np.array_equal(reference[1], candidate[1])),
    selected_method=method,
    selected_device=device,
    metrics={"trace_error": trace_error, "prediction_error": prediction_error, "predictions": candidate[1]},
)

TUTORIAL_RESULT::{"check": "training trace and predictions atol=8e-5", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"prediction_error": 1.1114630804609504e-07, "predictions": [0.9678730368614197, -0.9678730368614197, -0.9296988844871521, 0.9296988844871521], "trace_error": 1.565388414812713e-08}, "mettleq_median_ms": 166.6328124993015, "notebook": "pennylane/08_variational_classifier.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 69.27462501334958, "reference_over_mettleq": 0.4157321956840881, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}
